In [1]:
import cv2
import numpy as np
import mediapipe as mp
import os

DATA_PATH = "dataset"
SEQUENCE_LENGTH = 50

GESTURES = {
    ord('1'): "draw",
    ord('2'): "erase",
    ord('3'): "cursor",
    ord('4'): "scroll",
    ord('5'): "color",
    ord('6'): "screenshot",
    ord('7'): "clear"
}

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

cap = cv2.VideoCapture(0)

sequence = []
recording = False
label = "draw"

os.makedirs(DATA_PATH, exist_ok=True)

print("Controls: 1-7 select gesture | r start | s save | q quit")

def get_hand_data(results):
    left = np.zeros(63)
    right = np.zeros(63)

    if results.multi_hand_landmarks:
        for idx, hand_landmarks in enumerate(results.multi_hand_landmarks):

            hand = []

            for lm in hand_landmarks.landmark:
                hand.extend([lm.x, lm.y, lm.z])

            hand = np.array(hand)

            # assign first detected hand as left, second as right
            if idx == 0:
                left = hand
            elif idx == 1:
                right = hand

    return np.concatenate([left, right])  # 126 features

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

    if recording:
        frame_data = get_hand_data(results)
        sequence.append(frame_data)

    frame_count = len(sequence)
    progress = min(int((frame_count / SEQUENCE_LENGTH) * 100), 100)

    cv2.putText(frame, f"Gesture: {label}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.putText(frame, f"Recording: {recording}", (10, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.putText(frame, f"Frames: {frame_count}/{SEQUENCE_LENGTH}", (10, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    cv2.putText(frame, f"Progress: {progress}%", (10, 150),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    cv2.imshow("Two Hand Safe Collector", frame)

    key = cv2.waitKey(1) & 0xFF

    if key in GESTURES:
        label = GESTURES[key]
        print("Label:", label)

    elif key == ord('r'):
        recording = True
        sequence = []
        print("Recording started")

    elif key == ord('s') or len(sequence) >= SEQUENCE_LENGTH:
        recording = False

        if len(sequence) >= SEQUENCE_LENGTH:
            sequence = sequence[:SEQUENCE_LENGTH]

            save_path = os.path.join(DATA_PATH, label)
            os.makedirs(save_path, exist_ok=True)

            file_count = len(os.listdir(save_path))
            file_name = f"{file_count}.npy"

            np.save(os.path.join(save_path, file_name), sequence)

            print(f"Saved: {label}/{file_name} shape={np.array(sequence).shape}")
        else:
            print("Not enough frames")

        sequence = []

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Controls: 1-7 select gesture | r start | s save | q quit
Label: erase
Recording started
Saved: erase/0.npy shape=(50, 126)
Recording started
Saved: erase/1.npy shape=(50, 126)
Recording started
Saved: erase/2.npy shape=(50, 126)
Recording started
Saved: erase/3.npy shape=(50, 126)
Recording started
Saved: erase/4.npy shape=(50, 126)
Recording started
Saved: erase/5.npy shape=(50, 126)
Recording started
Saved: erase/6.npy shape=(50, 126)
Recording started
Saved: erase/7.npy shape=(50, 126)
Recording started
Saved: erase/8.npy shape=(50, 126)
Recording started
Saved: erase/9.npy shape=(50, 126)
Recording started
Saved: erase/10.npy shape=(50, 126)
Recording started
Saved: erase/11.npy shape=(50, 126)
Recording started
Saved: erase/12.npy shape=(50, 126)
Recording started
Saved: erase/13.npy shape=(50, 126)
Recording started
Saved: erase/14.npy shape=(50, 126)
Recording started
Saved: erase/15.npy shape=(50, 126)
Recording started
Saved: erase/16.npy shape=(50, 126)
Recording started
Save